# AUGURY — MiniCPM-V 4.6 vision fine-tune (Colab)

Cloud path for the AUGURY vision model. Uses **bf16 LoRA** (T4 16GB fits it comfortably — no QLoRA needed).

Guard rails: LLaMA-Factory (not Unsloth), transformers>=5.7.0, no packing, template `minicpm_v_4_6`, freeze vision tower, merge before GGUF.

**Dataset:** `data/vision/` from the Victus repo (see Setup below).

In [ ]:
# 0) GPU check
import torch
print('cuda:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > T4 GPU'

**Setup — get `data/vision/` into the runtime.** Two options:

**(A) Google Drive (recommended, survives session restarts):**
```python
from google.colab import drive
drive.mount('/content/drive')
# upload the repo's data/vision folder to Drive, e.g. MyDrive/augury_vision/data/vision
!mkdir -p /content/augury_vision
!ln -s /content/drive/MyDrive/augury_vision/data/vision /content/augury_vision/data 2>/dev/null || true
!cp /content/drive/MyDrive/augury_vision/train_v4_6_lora.yaml /content/augury_vision/ 2>/dev/null || true
```

**(B) Direct upload (no Drive):** use the Files panel / `files.upload()` to put the contents of `data/vision/` at `/content/augury_vision/data/vision` (i.e. images/, train.jsonl, val.jsonl, dataset_info.json, species_list.json) and `train_v4_6_lora.yaml` at `/content/augury_vision/`.

Either way the layout must be:
```
/content/augury_vision/
  data/vision/{images, train.jsonl, val.jsonl, dataset_info.json, ...}
  train_v4_6_lora.yaml
```

In [ ]:
# 1) Environment (pins per MiniCPM-V CookBook; T4 is CUDA 12.8-capable)
%pip install -q torch==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -q transformers==5.7.0 accelerate==1.13.0 peft==0.18.1 trl==0.24.0 \
    deepspeed==0.18.3 safetensors sentencepiece einops ninja torchaudio
!git clone -q https://github.com/hiyouga/LlamaFactory.git /content/LlamaFactory
%pip install -q -e /content/LlamaFactory
import transformers; assert transformers.__version__ >= '5.7.0', transformers.__version__
print('transformers', transformers.__version__)

In [ ]:
# 2) Model (download once per session) + patch the yaml paths
from huggingface_hub import snapshot_download
path = snapshot_download('openbmb/MiniCPM-V-4.6')
print('model at', path)

yaml_in = '/content/augury_vision/train_v4_6_lora.yaml'
yaml_out = '/content/augury_vision/train_v4_6_lora_colab.yaml'
txt = open(yaml_in).read()
txt = txt.replace('models/MiniCPM-V-4.6', path)                    # absolute model path
txt = txt.replace('dataset_dir: data/vision', 'dataset_dir: /content/augury_vision/data/vision')
txt = txt.replace('output_dir: data/vision/output', 'output_dir: /content/augury_vision/data/vision/output')
open(yaml_out, 'w').write(txt)
print('wrote', yaml_out)

# sanity: dataset files present?
import os
for f in ['data/vision/train.jsonl', 'data/vision/val.jsonl', 'data/vision/dataset_info.json']:
    p = f'/content/augury_vision/{f}'
    print(p, 'OK' if os.path.exists(p) else 'MISSING')
n_train = sum(1 for _ in open('/content/augury_vision/data/vision/train.jsonl'))
print('train rows:', n_train)

In [ ]:
# 3) Train (bf16 LoRA, ~3-4h on T4 for the full 58k-row dataset)
import os
os.environ['DOWNSAMPLE_MODE'] = '4x'      # guard rail 4 (fine detail; on T4 16GB this fits)
os.environ['DISABLE_VERSION_CHECK'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ.pop('USE_V1', None)            # v2 launcher
!cd /content/augury_vision && llamafactory-cli train train_v4_6_lora_colab.yaml 2>&1 | tail -60

**Checkpoints** land in `/content/augury_vision/data/vision/output/augury-v4_6-lora/`. The runtime may disconnect after ~12h (free) — fine, this run is ~3-4h. If interrupted, re-run cell 3: LLaMA-Factory resumes from the last checkpoint (`overwrite_output_dir: false` + trainer auto-resume only if you pass `--resume_from_checkpoint`; simplest is to copy checkpoints to Drive between runs).

In [ ]:
# 4) Merge LoRA into base fp16, then copy back to Drive
import os
os.environ['DISABLE_VERSION_CHECK'] = '1'
!cd /content/augury_vision && llamafactory-cli export \
  --model_name_or_path {path} \
  --adapter_name_or_path data/vision/output/augury-v4_6-lora \
  --template minicpm_v_4_6 --finetuning_type lora \
  --export_dir data/vision/output/augury-v4_6-merged --export_size 2
print('merged to /content/augury_vision/data/vision/output/augury-v4_6-merged')
# then copy to Drive:  !cp -r data/vision/output /content/drive/MyDrive/augury_vision/